In [5]:
import json
import re
import time
from huggingface_hub import hf_hub_download, list_models
import pandas as pd
import os
from dotenv import load_dotenv
import openai

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


def extract_hyperparams(text):
    system_prompt = "You are a data extraction bot. Given a Hugging Face model card text, extract the following fields if available: Training batch size, Evaluation batch size, Learning rate, Max training epochs, Max steps, Metric for best model, Word Error Rate (WER). Return the result as JSON without any extra text or explanation."
    user_prompt = f"Model Card:\n{text}\n\nExtract and format as JSON."

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo-1106",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
        )
        reply = response['choices'][0]['message']['content']

        # Try to find the JSON block inside the reply
        json_text_match = re.search(r"\{.*\}", reply, re.DOTALL)
        if not json_text_match:
            print("No JSON block found")
            return {}

        json_text = json_text_match.group(0)

        # Replace smart quotes with normal quotes
        json_text = json_text.replace("“", "\"").replace("”", "\"").replace("‘", "'").replace("’", "'")

        data = json.loads(json_text)
        return data

    except Exception as e:
        print(f"OpenAI extraction failed: {e}")
        return {}

# Step 1: Get models fine-tuned from whisper-small
models = list_models(
    search="whisper-small",
    sort="last_modified",
    limit=50,
    language="fr"
)

data = []
for model in models:
    print(f"Processing {model.modelId}...")
    try:
        # Download the README file manually
        readme_path = hf_hub_download(repo_id=model.modelId, filename="README.md")
        with open(readme_path, "r", encoding="utf-8") as f:
            text = f.read()

        if not text.strip():
            print(f"No README for {model.modelId}")
            continue

        params = extract_hyperparams(text)
        params["Model Name"] = model.modelId
        params["Model Link"] = f"https://huggingface.co/{model.modelId}"
        params["Likes"] = model.likes
        params["Downloads"] = model.downloads
        data.append(params)

        time.sleep(1)

    except Exception as e:
        print(f"Failed {model.modelId}: {e}")
        continue

import pandas as pd
df = pd.DataFrame(data)
df.to_csv("whisper_small_finetunes_summary.csv", index=False)
print("✅ Saved to whisper_small_finetunes_summary.csv")

Processing hellomefriend/whisper-small-dv...
Processing visalkao/whisper-small-french-finetuning...
Processing mozilla-ai/whisper-small-fr...
Processing lordyhas/whisper-small-fr...
Processing TaphaFall/whisper-small-wo-final...
Processing ngia/whisper-small-wolof-v2...
Processing ngia/whisper-small-wo...
Processing deepdml/whisper-small-mix-fr...
Processing Intel/whisper-small-openvino...
OpenAI extraction failed: This model's maximum context length is 16385 tokens. However, your messages resulted in 26645 tokens. Please reduce the length of the messages.
Processing smrc/new-whisper-small-fr-qc...
Processing Yujiyakov/whisper-small...
Processing Hanhpt23/whisper-small-frenchmed-free_E0-8D3-11...
Processing Hanhpt23/whisper-small-frenchmed-free_E3-11...
Processing Hanhpt23/whisper-small-frenchmed-free_E0-8...
Processing Hanhpt23/whisper-small-frenchmed-free_E0-8D0-8...
Processing Hanhpt23/whisper-small-frenchmed-free_ED0-8...
Processing Hanhpt23/whisper-small-frenchmed-free_ED3-11...
P

In [6]:
import pandas as pd
import numpy as np

df['Training batch size'] = pd.to_numeric(df['Training batch size'], errors='coerce')
df['Evaluation batch size'] = pd.to_numeric(df['Evaluation batch size'], errors='coerce')
df['Learning rate'] = pd.to_numeric(df['Learning rate'], errors='coerce')
df['Max training epochs'] = pd.to_numeric(df['Max training epochs'], errors='coerce')
df['Max steps'] = pd.to_numeric(df['Max steps'], errors='coerce')

df['Metric for best model'] = df['Metric for best model'].replace({
    'Wer': 'wer',
    'Word Error Rate (WER)': 'wer',
    'Test WER': 'wer',
    'WER': 'wer',
    'eval_wer': 'wer'
})
def clean_wer(value):
    if pd.isna(value) or value == "":
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, dict):
        return np.nan
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan


def extract_wer_from_metrics(metrics):
    if isinstance(metrics, str) and "Word Error Rate (WER)" in metrics:
        try:
            import ast
            metrics_dict = ast.literal_eval(metrics)
            if isinstance(metrics_dict, dict):
                return list(metrics_dict.values())[0] if metrics_dict else np.nan
        except:
            return np.nan
    return np.nan

df['Word Error Rate (WER)'] = df['Word Error Rate (WER)'].apply(clean_wer)
df['metrics'] = df['metrics'].replace('', np.nan)
missing_wer_mask = df['Word Error Rate (WER)'].isna()
df.loc[missing_wer_mask, 'Word Error Rate (WER)'] = df.loc[missing_wer_mask, 'metrics'].apply(extract_wer_from_metrics)
df = df.dropna(subset=['Model Name'])
df['Word Error Rate (WER)'] = pd.to_numeric(df['Word Error Rate (WER)'], errors='coerce')


In [7]:
df.to_csv("whisper_small_finetunes_summary_cleaned.csv", index=False)